# Server-side temporal aggregation with `aggregate=`

On openEO, `aggregate=AggregationConfig(...)` is a **native** `aggregate_temporal_period` node in the graph - the only backend where the aggregator costs no client compute. Here we load a plain Sentinel-2 collection (B04, B08) over a year and reduce it to a **monthly mean** server-side. The aggregator's pandas `freq` maps to an openEO calendar `period` and its `op` to an openEO reducer.

```bash
pip install earthlens[openeo]
```

## Credentials

openEO authenticates with CDSE via OIDC. The download cell below runs for real when credentials are present (env `OPENEO_CLIENT_ID` / `OPENEO_CLIENT_SECRET`, or a cached refresh token from a prior interactive login) and prints a skip note otherwise - so this notebook executes cleanly with or without secrets. See the Authentication page.

In [ ]:
import os
from pathlib import Path


def has_openeo_credentials() -> bool:
    """Whether some openEO OIDC credential source is available."""
    if os.environ.get("OPENEO_CLIENT_ID") and os.environ.get("OPENEO_CLIENT_SECRET"):
        return True
    if os.environ.get("OPENEO_REFRESH_TOKEN"):
        return True
    return (Path.home() / ".openeo").exists()


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    CDSE OIDC credentials are configured (so the docs build never needs secrets).
    """
    if not has_openeo_credentials():
        print(
            "No openEO OIDC credentials found - skipping the live download.\n"
            "Set OPENEO_CLIENT_ID / OPENEO_CLIENT_SECRET (headless) or sign in once\n"
            "interactively (see the Authentication page) to run this cell for real."
        )
        return []
    paths = facade.download(**download_kwargs)
    for path in paths:
        print(path)
    return paths


## Build the request

In [ ]:
from earthlens import EarthLens
from earthlens.aggregate import AggregationConfig

facade = EarthLens(
    data_source='openeo',
    variables={'sentinel-2-l2a': ['B04', 'B08']},
    start='2023-01-01', end='2023-12-31',
    lat_lim=[40.40, 40.45], lon_lim=[3.67, 3.72],
    path='data/openeo-aggregate',
    max_cloud_cover=40,
)
config = AggregationConfig(freq='1MS', op='mean')
config

The mapping is `1MS -> month` and `mean -> mean`:

In [ ]:
from earthlens.openeo._helpers import period_for, reducer_for
print('period:', period_for(config.freq))
print('reducer:', reducer_for(config.op))

## Download (server-side execution)

In [ ]:
paths = run_or_skip(facade, aggregate=config)
paths